    
### DFL 전체 파이프라인 (Forward Pass)

    Args:
        z          : (batch, input_dim)  - 입력 피처
        r_real     : (batch, N, m)       - 실현 수익률 (ground truth)
        pred_model : PredictionModel
        opt_layer  : CvxpyLayer
        n1         : drawdown 한도 비율
        C          : 자본 규모
        d          : 연수
        x_min      : 최소 비중
        x_max      : 최대 비중
        lam        : MDD 패널티 가중치
    
    Returns:
        dict with keys:
            r_hat   : (batch, N, m)  예측 수익률
            y_hat   : (batch, N, m)  예측 누적 수익률
            x_star  : (batch, m)     최적 포트폴리오
            y_real  : (batch, N, m)  실현 누적 수익률
            w_real  : (batch, N)     실현 포트폴리오 경로
            R_real  : (batch,)       실현 수익률
            M_real  : (batch,)       실현 MDD
            loss    : scalar         DFL Loss

In [10]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import cvxpy as cp
from cvxpylayers.torch import CvxpyLayer
import yfinance as yf
import random
from dataclasses import dataclass
from typing import Dict, Tuple
from tqdm import tqdm
import itertools
import torch.nn as nn
import torch.optim as optim


    
    
### MDD Formulation을 cvxpylayers로 구성

    목적함수:
        max  (1/dC) * y_hat(N)^T x
        (dC는 상수이므로 y_hat(N)^T x 최대화와 동치)

    제약식:
        u_k - y_hat_k^T x <= n1 * C,    k = 1,...,N   (drawdown 상한)
        u_k >= y_hat_k^T x,              k = 1,...,N   (running max >= 현재값)
        u_k >= u_{k-1},                  k = 1,...,N   (running max 단조증가)
        u_0 = 0
        x_min <= x_i <= x_max,          i = 1,...,m   (box constraint)

    Parameters (cvxpy):
        Y_hat : (N, m) - 예측 누적 수익률 경로 (각 행이 y_hat(t))
        n1C   : scalar - 허용 최대 drawdown 한도 (n1 * C)
        x_min : scalar
        x_max : scalar

    Returns:
        CvxpyLayer - differentiable optimization layer
    

In [11]:
np.random.seed(42)

n_stocks = 10
n_days = 252
stock_names = [f"S{i+1}" for i in range(n_stocks)]

# Random daily returns
mu = np.random.uniform(0.001, 0.01, n_stocks)
sigma = np.random.uniform(0.01, 0.05, n_stocks)
returns = np.random.normal(mu, sigma, size=(n_days, n_stocks))

dates = pd.bdate_range(end="2025-12-31", periods=n_days)
returns_df = pd.DataFrame(returns, index=dates, columns=stock_names)
prices_df = (1 + returns_df).cumprod() * 100  # start at 100

# Period split
is_end = dates[int(n_days * 6 / 12) - 1]   # end of IS (first 6M)
oos_start = dates[int(n_days * 6 / 12)]     # start of OOS (last 6M)

# Backtesting window: last 6 months of total period
bt_start = dates[int(n_days * 6 / 12)]
bt_end   = dates[-1]

# Rebalancing dates: monthly within backtest window
rebal_dates = pd.bdate_range(bt_start, bt_end, freq="BME")  # Business Month End

print(f"Total period  : {dates[0].date()} → {dates[-1].date()} ({n_days} days)")
print(f"IS            : {dates[0].date()} → {is_end.date()}")
print(f"OOS / BT start: {oos_start.date()} → {bt_end.date()}")
print(f"Rebal dates   : {[d.date() for d in rebal_dates]}")
print(f"\nReturns shape : {returns_df.shape}")
print(returns_df.head())


Total period  : 2025-01-14 → 2025-12-31 (252 days)
IS            : 2025-01-14 → 2025-07-08
OOS / BT start: 2025-07-09 → 2025-12-31
Rebal dates   : [datetime.date(2025, 7, 31), datetime.date(2025, 8, 29), datetime.date(2025, 9, 30), datetime.date(2025, 10, 31), datetime.date(2025, 11, 28), datetime.date(2025, 12, 31)]

Returns shape : (252, 10)
                  S1        S2        S3        S4        S5        S6  \
2025-01-14 -0.006591  0.024891 -0.031727 -0.019731  0.027720 -0.001510   
2025-01-15 -0.008087  0.027889 -0.018418  0.000993 -0.007989  0.034515   
2025-01-16  0.006631 -0.086068 -0.049919  0.010029  0.015160  0.005375   
2025-01-17 -0.000615  0.061140  0.022466 -0.026217  0.008002 -0.004272   
2025-01-20 -0.004712 -0.005532  0.021931  0.024429 -0.005873 -0.000815   

                  S7        S8        S9       S10  
2025-01-14  0.003020 -0.035358 -0.008440  0.009774  
2025-01-15  0.001224 -0.023983  0.028847 -0.019058  
2025-01-16 -0.001041 -0.000536 -0.033921 -0.008211

In [12]:


# =============================================================================
# Step 1. Prediction Model: z -> r_hat
# =============================================================================

class PredictionModel(nn.Module):
   
    def __init__(self, input_dim: int, hidden_dim: int, N: int, m: int):
        super().__init__()
        self.N = N
        self.m = m
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, N * m),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
  
        batch = z.shape[0]
        out = self.net(z)               # (batch, N*m)
        r_hat = out.view(batch, self.N, self.m)
        return r_hat


# =============================================================================
# Step 2. Cumulative Return Path: r_hat -> y_hat
# =============================================================================

def compute_cumulative_path(r: torch.Tensor) -> torch.Tensor:
  
    y = torch.cumsum(r, dim=1)  # t축으로 누적합
    return y


# =============================================================================
# Step 3. Optimization Layer: y_hat -> x_star  (Formulation 16)
# =============================================================================

def build_optimization_layer(N: int, m: int) -> CvxpyLayer:
   
    # --- cvxpy 변수 ---
    x = cp.Variable(m, name="x")           # 포트폴리오 가중치 (m,)
    u = cp.Variable(N + 1, name="u")       # running max 보조변수 (N+1,) : u[0],...,u[N]

    # --- cvxpy 파라미터 ---
    Y_hat = cp.Parameter((N, m), name="Y_hat")   # 예측 누적 수익률 경로
    n1C   = cp.Parameter(nonneg=True, name="n1C") # drawdown 한도
    x_min = cp.Parameter(name="x_min")
    x_max = cp.Parameter(name="x_max")

    # --- 목적함수: y_hat(N)^T x 최대화 ---
    # Y_hat[N-1] == y_hat(t=N) (0-indexed)
    objective = cp.Maximize(Y_hat[N - 1] @ x)

    # --- 제약식 ---
    constraints = []

    # u_0 = 0
    constraints.append(u[0] == 0)

    for k in range(1, N + 1):
        y_k = Y_hat[k - 1]          # y_hat(t=k), shape (m,)

        # drawdown 제약: u_k - y_k^T x <= n1C
        constraints.append(u[k] - y_k @ x <= n1C)

        # running max >= 현재 포트폴리오 cumulative return
        constraints.append(u[k] >= y_k @ x)

        # running max 단조증가
        constraints.append(u[k] >= u[k - 1])

    # box constraint
    constraints.append(x >= x_min)
    constraints.append(x <= x_max)
    constraints.append(cp.sum(x) == 1)

    # --- 문제 정의 ---
    problem = cp.Problem(objective, constraints)
    assert problem.is_dcp(), "Problem is not DCP!"

    # --- CvxpyLayer 생성 ---
    layer = CvxpyLayer(
        problem,
        parameters=[Y_hat, n1C, x_min, x_max],
        variables=[x, u],
    )
    return layer


def solve_portfolio(
    y_hat: torch.Tensor,
    opt_layer: CvxpyLayer,
    n1: float,
    C: float,
    x_min: float,
    x_max: float,
) -> torch.Tensor:
    
    batch, N, m = y_hat.shape

    n1C_val   = torch.tensor(n1 * C, dtype=torch.float64)
    x_min_val = torch.tensor(x_min, dtype=torch.float64)
    x_max_val = torch.tensor(x_max, dtype=torch.float64)

    x_stars = []
    for b in range(batch):
        Y_hat_b = y_hat[b].double()  # (N, m)
        x_star_b, _ = opt_layer(
            Y_hat_b,
            n1C_val,
            x_min_val,
            x_max_val,
            solver_args={"solve_method": "ECOS"},
        )
        x_stars.append(x_star_b.float())

    x_star = torch.stack(x_stars, dim=0)  # (batch, m)
    return x_star


# =============================================================================
# Step 4. Realized Portfolio Path: (x_star, y_real) -> w_real(t)
# =============================================================================

def compute_realized_path(
    x_star: torch.Tensor,
    y_real: torch.Tensor,
) -> torch.Tensor:

    # einsum: w_real[b,t] = sum_j x_star[b,j] * y_real[b,t,j]
    w_real = torch.einsum("bj, btj -> bt", x_star, y_real)  # (batch, N)
    return w_real


# =============================================================================
# Step 5. Performance Metrics: w_real -> R_real, M_real
# =============================================================================

def compute_return(
    w_real: torch.Tensor,
    d: float,
    C: float,
) -> torch.Tensor:
    
    R_real = w_real[:, -1] / (d * C)
    return R_real


def compute_max_drawdown(w_real: torch.Tensor) -> torch.Tensor:
    
    # running maximum up to each time step
    running_max, _ = torch.cummax(w_real, dim=1)  # (batch, N)

    # drawdown at each time step
    drawdown = running_max - w_real                # (batch, N)

    # maximum drawdown
    M_real = torch.max(drawdown, dim=1).values 
    # M_real = torch.logsumexp(beta * drawdown, dim=1) / beta   # (Gradient가 smooth 하지 않을 때 사용)
    return M_real



# def compute_max_drawdown_smoothing(w_real: torch.Tensor, beta: float = 100.0) -> torch.Tensor:
#     """
#     실현 MaxDD 계산 (differentiable 근사)

#     D_real(t) = max_{tau <= t} w_real(tau) - w_real(t)
#     M_real    = max_{1 <= t <= N} D_real(t)

#     Note:
#         torch.cummax는 미분 가능하지만 max는 argmax가 겹치면
#         gradient가 불안정할 수 있음.
#         DFL loss의 M_real 항은 gradient를 통해 theta를 업데이트하는 데 사용.

#     Args:
#         w_real : (batch, N)
#     Returns:
#         M_real : (batch,)
#     """
#     # running maximum up to each time step
#     running_max, _ = torch.cummax(w_real, dim=1)  # (batch, N)

#     # drawdown at each time step
#     drawdown = running_max - w_real                # (batch, N)

#     # maximum drawdown
#     # M_real = torch.max(drawdown, dim=1).values 
#     M_real = torch.logsumexp(beta * drawdown, dim=1) / beta   (Gradient가 smooth 하지 않을 때 사용)
#     return M_real



# =============================================================================
# Step 6. DFL Loss Function
# =============================================================================

def dfl_loss(
    R_real: torch.Tensor,
    M_real: torch.Tensor,
    lam: float,
) -> torch.Tensor:
   
    loss = (-R_real + lam * M_real).mean()
    return loss


# =============================================================================
# Full Pipeline (End-to-End)
# =============================================================================

def forward_pass(
    z: torch.Tensor,
    r_real: torch.Tensor,
    pred_model: PredictionModel,
    opt_layer: CvxpyLayer,
    n1: float,
    C: float,
    d: float,
    x_min: float,
    x_max: float,
    lam: float,
) -> dict:

    # Step 1: z -> r_hat
    r_hat = pred_model(z)                           # (batch, N, m)

    # Step 2: r_hat -> y_hat
    y_hat = compute_cumulative_path(r_hat)          # (batch, N, m)

    # Step 3: y_hat -> x_star  (LP)
    x_star = solve_portfolio(
        y_hat, opt_layer, n1, C, x_min, x_max
    )                                               # (batch, m)

    # Step 4: x_star + y_real -> w_real
    y_real = compute_cumulative_path(r_real)        # (batch, N, m)
    w_real = compute_realized_path(x_star, y_real)  # (batch, N)

    # Step 5: w_real -> R_real, M_real
    R_real = compute_return(w_real, d, C)           # (batch,)
    M_real = compute_max_drawdown(w_real)           # (batch,)

    # Step 6: Loss
    loss = dfl_loss(R_real, M_real, lam)            # scalar

    return {
        "r_hat" : r_hat,
        "y_hat" : y_hat,
        "x_star": x_star,
        "y_real": y_real,
        "w_real": w_real,
        "R_real": R_real,
        "M_real": M_real,
        "loss"  : loss,
    }

In [14]:

np.random.seed(42)
torch.manual_seed(42)

N_STOCKS   = 10
N_DAYS     = 252
LOOKBACK   = 21   # feature window  (past 1M of returns, flattened)
HORIZON    = 21   # prediction & rebalancing horizon (1M)

INPUT_DIM  = LOOKBACK * N_STOCKS   # 210
HIDDEN_DIM = 64
N          = HORIZON               # used inside framework
M          = N_STOCKS

n1    = 0.10   # drawdown limit factor
C     = 1.0    # initial capital
d     = float(N)   # return normalizer
x_min = 0.0    # long-only
x_max = 0.30   # max 30% per stock
lam   = 1.0    # loss weight on MDD

EPOCHS     = 50
BATCH_SIZE = 16
LR         = 1e-3

# =============================================================================
# 1. Random Data  (same as before)
# =============================================================================
stock_names = [f"S{i+1}" for i in range(N_STOCKS)]
mu    = np.random.uniform(0.0002, 0.001, N_STOCKS)
sigma = np.random.uniform(0.01,   0.03,  N_STOCKS)
returns_np  = np.random.normal(mu, sigma, size=(N_DAYS, N_STOCKS))  # (252, 10)

dates       = pd.bdate_range(end="2025-12-31", periods=N_DAYS)
returns_df  = pd.DataFrame(returns_np, index=dates, columns=stock_names)

IS_END_IDX  = N_DAYS // 2   # 126  ── IS/OOS boundary
BT_START    = IS_END_IDX    # backtest starts here

print(f"IS  : {dates[0].date()} → {dates[IS_END_IDX-1].date()}  ({IS_END_IDX} days)")
print(f"BT  : {dates[BT_START].date()} → {dates[-1].date()}  ({N_DAYS - BT_START} days)")

# =============================================================================
# 2. Sliding-Window Dataset
#    z      = returns[t-LOOKBACK : t].flatten()    (input_dim,)
#    r_real = returns[t : t+HORIZON]               (N, m)
# =============================================================================
def make_windows(arr, lookback, horizon, start, end):
    samples = []
    for t in range(start, end - horizon + 1):
        z      = arr[t - lookback : t].flatten()   # (lookback * m,)
        r_real = arr[t : t + horizon]              # (N, m)
        samples.append((z, r_real))
    return samples

# IS samples  (need lookback history → start from index LOOKBACK)
is_samples    = make_windows(returns_np, LOOKBACK, HORIZON,
                              start=LOOKBACK, end=IS_END_IDX)

# BT samples  (all windows in OOS, step by step)
bt_samples_all = make_windows(returns_np, LOOKBACK, HORIZON,
                               start=BT_START, end=N_DAYS)

# Monthly rebalancing: one window per HORIZON days
rebal_samples = bt_samples_all[::HORIZON]

print(f"\nIS training windows : {len(is_samples)}")
print(f"BT rebal windows    : {len(rebal_samples)}")

# =============================================================================
# 3. Instantiate Model & Optimization Layer
# =============================================================================
pred_model = PredictionModel(INPUT_DIM, HIDDEN_DIM, N, M)
opt_layer  = build_optimization_layer(N, M)
optimizer  = optim.Adam(pred_model.parameters(), lr=LR)

def to_tensors(samples):
    zs      = torch.tensor(np.array([s[0] for s in samples]), dtype=torch.float32)
    r_reals = torch.tensor(np.array([s[1] for s in samples]), dtype=torch.float32)
    return zs, r_reals   # (B, input_dim),  (B, N, m)

zs_is, rs_is = to_tensors(is_samples)

# =============================================================================
# 4. IS Training
# =============================================================================
print("\n── IS Training ──")
pred_model.train()

for epoch in range(EPOCHS):
    perm   = torch.randperm(len(is_samples))
    ep_loss = []

    for i in range(0, len(is_samples), BATCH_SIZE):
        idx = perm[i : i + BATCH_SIZE]
        z_b = zs_is[idx]
        r_b = rs_is[idx]

        optimizer.zero_grad()
        result = forward_pass(z_b, r_b, pred_model, opt_layer,
                              n1, C, d, x_min, x_max, lam)
        result["loss"].backward()
        optimizer.step()
        ep_loss.append(result["loss"].item())

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d}/{EPOCHS}  loss = {np.mean(ep_loss):.6f}")

# =============================================================================
# 5. OOS Backtesting  (walk-forward, no retraining)
# =============================================================================
pred_model.eval()
bt_results = []

print("\n── Backtest (walk-forward, monthly rebalancing) ──")
print(f"{'Win':>4}  {'R_real':>8}  {'MDD':>8}  {'Top-3 weights'}")
print("-" * 55)

for i, (z_np, r_np) in enumerate(rebal_samples):
    z      = torch.tensor(z_np[None],  dtype=torch.float32)   # (1, input_dim)
    r_real = torch.tensor(r_np[None],  dtype=torch.float32)   # (1, N, m)

    with torch.no_grad():
        r_hat = pred_model(z)                        # (1, N, m)

    y_hat  = compute_cumulative_path(r_hat)          # (1, N, m)  — needs grad for layer
    x_star = solve_portfolio(y_hat.detach(), opt_layer, n1, C, x_min, x_max)  # (1, m)

    y_real = compute_cumulative_path(r_real)
    w_real = compute_realized_path(x_star, y_real)   # (1, N)
    R_real = compute_return(w_real, d, C)
    M_real = compute_max_drawdown(w_real)

    w = x_star[0].numpy()
    top3 = {stock_names[j]: round(w[j], 3)
            for j in np.argsort(w)[-3:][::-1]}

    bt_results.append({
        "window": i + 1,
        "weights": w,
        "R_real":  R_real[0].item(),
        "M_real":  M_real[0].item(),
    })
    print(f"  {i+1:2d}  {R_real[0].item():8.4f}  {M_real[0].item():8.4f}  {top3}")

# =============================================================================
# 6. Summary
# =============================================================================
R_list = [r["R_real"] for r in bt_results]
M_list = [r["M_real"] for r in bt_results]

print("\n── Backtest Summary ──")
print(f"  Avg Return   : {np.mean(R_list):.4f}")
print(f"  Total Return : {np.sum(R_list):.4f}")
print(f"  Avg MDD      : {np.mean(M_list):.4f}")
print(f"  Max MDD      : {np.max(M_list):.4f}")
print(f"  Calmar Ratio : {np.sum(R_list) / (np.max(M_list) + 1e-8):.4f}")


IS  : 2025-01-14 → 2025-07-08  (126 days)
BT  : 2025-07-09 → 2025-12-31  (126 days)

IS training windows : 85
BT rebal windows    : 6

── IS Training ──
  Epoch   1/50  loss = 0.038432
  Epoch   5/50  loss = 0.042981
  Epoch  10/50  loss = 0.043337
  Epoch  15/50  loss = 0.043124
  Epoch  20/50  loss = 0.043628
  Epoch  25/50  loss = 0.041181
  Epoch  30/50  loss = 0.040735
  Epoch  35/50  loss = 0.040113
  Epoch  40/50  loss = 0.041154
  Epoch  45/50  loss = 0.040787
  Epoch  50/50  loss = 0.041816

── Backtest (walk-forward, monthly rebalancing) ──
 Win    R_real       MDD  Top-3 weights
-------------------------------------------------------
   1    0.0073    0.0106  {'S7': 0.3, 'S6': 0.3, 'S2': 0.3}
   2    0.0025    0.0244  {'S7': 0.3, 'S6': 0.3, 'S2': 0.3}
   3    0.0024    0.0241  {'S7': 0.3, 'S6': 0.3, 'S2': 0.3}
   4    0.0001    0.0437  {'S7': 0.3, 'S6': 0.3, 'S2': 0.3}
   5    0.0014    0.0285  {'S7': 0.3, 'S6': 0.3, 'S2': 0.3}
   6   -0.0009    0.0720  {'S7': 0.3, 'S6': 0.3